In [1]:
import torch

# preprocessor
processor = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_preprocessor')
# models
vjepa2_vit_large = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_large')
# vjepa2_vit_huge = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_huge')
# vjepa2_vit_giant = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant')
# vjepa2_vit_giant_384 = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')



Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main
Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main
/home/wjl/.conda/envs/jepa_torch/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:

vjepa2_vit_large[0]




VisionTransformer(
  (patch_embed): PatchEmbed3D(
    (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (blocks): ModuleList(
    (0-23): 24 x Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
)

In [4]:
vjepa2_vit_large[1]


VisionTransformerPredictor(
  (predictor_embed): Linear(in_features=1024, out_features=384, bias=True)
  (mask_tokens): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x1x384]
      (1): Parameter containing: [torch.float32 of size 1x1x384]
      (2): Parameter containing: [torch.float32 of size 1x1x384]
      (3): Parameter containing: [torch.float32 of size 1x1x384]
      (4): Parameter containing: [torch.float32 of size 1x1x384]
      (5): Parameter containing: [torch.float32 of size 1x1x384]
      (6): Parameter containing: [torch.float32 of size 1x1x384]
      (7): Parameter containing: [torch.float32 of size 1x1x384]
      (8): Parameter containing: [torch.float32 of size 1x1x384]
      (9): Parameter containing: [torch.float32 of size 1x1x384]
  )
  (predictor_blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=384, out_features=115

In [12]:
import torch

data = torch.randn(8, 3, 6, 256, 256)

out = vjepa2_vit_large[0](data)

/home/wjl/.conda/envs/jepa_torch/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [13]:

out.shape


torch.Size([8, 768, 1024])

In [14]:
768/6

128.0

In [ ]:
import json

path = "data/ssv2/labels/train.json"

with open(path, "r") as f:
    data = json.load(f)

print(data)





In [ ]:
data

In [ ]:
import torch

ckpt = torch.load("/data/wjl/vjepa2/logs/e0.pt", map_location="cpu")
print(ckpt.keys())



In [ ]:
ckpt["target_encoder"]

In [ ]:
import sys
sys.path.append("/data/wjl/vjepa2")

from src.models.vision_transformer import vit_huge
import torch.nn as nn
from functools import partial

model = vit_huge()

model
 




In [ ]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader



from app.vjepa.utils import init_opt, init_video_model, load_checkpoint

encoder, predictor = init_video_model(
    device="cpu",
    patch_size=16,
    max_num_frames=16,
    tubelet_size=2,
    model_name="vit_large",
    crop_size=224,
    pred_depth=6,
    pred_num_heads=None,
    pred_embed_dim=384,
    uniform_power=False,
    use_mask_tokens=True,
    num_mask_tokens=2,
    zero_init_mask_tokens=True,
    use_sdpa=False,
    use_rope=True,
    use_silu=False,
    use_pred_silu=False,
    wide_silu=False,
    use_activation_checkpointing=False,
)


r_path = "/data/wjl/vjepa2/ckpts/vitl.pt"

print(f"Loading checkpoint from {r_path}")
checkpoint = robust_checkpoint_loader(r_path, map_location=torch.device("cpu"))

epoch = checkpoint["epoch"]

# -- loading encoder
pretrained_dict = checkpoint["encoder"]

# 处理键名，移除'module.'前缀
new_pretrained_dict = {}
for k, v in pretrained_dict.items():
    if k.startswith('module.'):
        new_k = k[7:]  # 移除'module.'前缀
        new_pretrained_dict[new_k] = v
    else:
        new_pretrained_dict[k] = v

msg = encoder.load_state_dict(new_pretrained_dict, strict=False)
print(f"loaded pretrained encoder from epoch {epoch} with msg: {msg}")





In [ ]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader
from app.vjepa.utils import init_video_model

encoder, predictor = init_video_model(
    device="cpu",
    patch_size=16,
    max_num_frames=16,
    tubelet_size=2,
    model_name="vit_large",
    crop_size=224,
    pred_depth=6,
    pred_num_heads=None,
    pred_embed_dim=384,
    uniform_power=False,
    use_mask_tokens=True,
    num_mask_tokens=2,
    zero_init_mask_tokens=True,
    use_sdpa=False,
    use_rope=True,
    use_silu=False,
    use_pred_silu=False,
    wide_silu=False,
    use_activation_checkpointing=False,
)

r_path = "/data/wjl/vjepa2/ckpts/vitl.pt"
print(f"Loading checkpoint from {r_path}")
checkpoint = robust_checkpoint_loader(r_path, map_location=torch.device("cpu"))
epoch = checkpoint["epoch"]

# checkpoint encoder权重
pretrained_dict = checkpoint["encoder"]
encoder_dict = encoder.state_dict()
predictor_dict = predictor.state_dict()

pretrained_encoder_dict = {}
pretrained_predictor_dict = {}

# 处理checkpoint的key，分离encoder与predictor，并去除多余的前缀
for k, v in pretrained_dict.items():
    # 1. 移除module.前缀
    new_k = k[7:] if k.startswith('module.') else k

    # 2. 检查是否属于predictor
    if new_k.startswith("predictor."):
        predictor_key = new_k[len("predictor."):]  # 去掉predictor.前缀

        # 3. 再次检查是否有backbone.前缀
        if predictor_key.startswith("backbone."):
            predictor_key = predictor_key[len("backbone."):]  # 去掉backbone.前缀

        if predictor_key in predictor_dict:
            pretrained_predictor_dict[predictor_key] = v
        else:
            print(f"Predictor key not matched in model: {predictor_key}")
    else:
        # encoder的权重
        if new_k in encoder_dict:
            pretrained_encoder_dict[new_k] = v
        else:
            print(f"Encoder key not matched in model: {new_k}")

# 加载encoder权重
encoder_msg = encoder.load_state_dict(pretrained_encoder_dict, strict=False)
print(f"Loaded pretrained encoder from epoch {epoch} with msg: {encoder_msg}")

# 加载predictor权重
predictor_msg = predictor.load_state_dict(pretrained_predictor_dict, strict=False)
print(f"Loaded pretrained predictor from epoch {epoch} with msg: {predictor_msg}")




In [2]:
import torch

vjepa2_encoder, vjepa2_ac_predictor = torch.hub.load('facebookresearch/vjepa2', 
                                                     'vjepa2_ac_vit_giant')





  8%|▊         | 880M/11.0G [01:10<13:44, 13.1MB/s]


KeyboardInterrupt: 

In [3]:
torch.hub.list('facebookresearch/vjepa2')

Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main


['vjepa2_ac_vit_giant',
 'vjepa2_preprocessor',
 'vjepa2_vit_giant',
 'vjepa2_vit_giant_384',
 'vjepa2_vit_huge',
 'vjepa2_vit_large']

In [ ]:
# 假设你有本地的 hubconf.py 文件
model = torch.hub.load('/data/wjl/vjepa2/logs/e0.pt', 'vjepa2_vit_large', source='local')




In [26]:

import torch

model = torch.load("/data/wjl/vjepa2/logs/32.8.vitl16-256px-64f_cooldown/latest.pt", map_location="cpu")




In [28]:
model.keys()

dict_keys(['encoder', 'predictor', 'opt', 'scaler', 'target_encoder', 'epoch', 'loss', 'batch_size', 'world_size', 'lr'])

In [30]:
model["encoder"].keys()

odict_keys(['module.backbone.patch_embed.proj.weight', 'module.backbone.patch_embed.proj.bias', 'module.backbone.blocks.0.norm1.weight', 'module.backbone.blocks.0.norm1.bias', 'module.backbone.blocks.0.attn.qkv.weight', 'module.backbone.blocks.0.attn.qkv.bias', 'module.backbone.blocks.0.attn.proj.weight', 'module.backbone.blocks.0.attn.proj.bias', 'module.backbone.blocks.0.norm2.weight', 'module.backbone.blocks.0.norm2.bias', 'module.backbone.blocks.0.mlp.fc1.weight', 'module.backbone.blocks.0.mlp.fc1.bias', 'module.backbone.blocks.0.mlp.fc2.weight', 'module.backbone.blocks.0.mlp.fc2.bias', 'module.backbone.blocks.1.norm1.weight', 'module.backbone.blocks.1.norm1.bias', 'module.backbone.blocks.1.attn.qkv.weight', 'module.backbone.blocks.1.attn.qkv.bias', 'module.backbone.blocks.1.attn.proj.weight', 'module.backbone.blocks.1.attn.proj.bias', 'module.backbone.blocks.1.norm2.weight', 'module.backbone.blocks.1.norm2.bias', 'module.backbone.blocks.1.mlp.fc1.weight', 'module.backbone.blocks.1

In [32]:
model["scaler"].keys()




dict_keys(['scale', 'growth_factor', 'backoff_factor', 'growth_interval', '_growth_tracker'])

In [54]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader
from app.vjepa.utils import init_video_model
import copy

encoder, predictor = init_video_model(
    device="cpu",
    patch_size=16,
    max_num_frames=16,
    tubelet_size=2,
    model_name="vit_large",
    crop_size=224,
    pred_depth=12,
    pred_num_heads=None,
    pred_embed_dim=384,
    uniform_power=False,
    use_mask_tokens=True,
    num_mask_tokens=10,
    zero_init_mask_tokens=True,
    use_sdpa=False,
    use_rope=True,
    use_silu=False,
    use_pred_silu=False,
    wide_silu=False,
    use_activation_checkpointing=False,
)


target_encoder = copy.deepcopy(encoder)



[INFO    ][2025-07-09 16:38:15][root                ][init_video_model         ] MultiSeqWrapper(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): RoPEAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0

In [37]:

def load_pretrained(encoder, pretrained, checkpoint_key="target_encoder"):
    print(f"Loading pretrained model from {pretrained}")
    checkpoint = robust_checkpoint_loader(pretrained, map_location="cpu")
    try:
        pretrained_dict = checkpoint[checkpoint_key]
    except Exception:
        pretrained_dict = checkpoint["encoder"]

    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
#     pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    for k, v in encoder.state_dict().items():
        if k not in pretrained_dict:
            print(f"key '{k}' could not be found in loaded state dict")
        elif pretrained_dict[k].shape != v.shape:
            print(f"{pretrained_dict[k].shape} | {v.shape}")
            print(f"key '{k}' is of different shape in model and loaded state dict")
            exit(1)
            pretrained_dict[k] = v
    msg = encoder.load_state_dict(pretrained_dict, strict=False)
    print(encoder)
    print(f"loaded pretrained model with msg: {msg}")
    print(f"loaded pretrained encoder from epoch: {checkpoint['epoch']}\n path: {pretrained}")
    del checkpoint
    return encoder

pretrained = "/data/wjl/vjepa2/ckpts/vitl.pt"
encoder = load_pretrained(encoder, pretrained, checkpoint_key="encoder")



Loading pretrained model from /data/wjl/vjepa2/ckpts/vitl.pt


MultiSeqWrapper(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): RoPEAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((1024,), eps=1e-

In [ ]:

def load_any_pretrained(encoder, pretrained, checkpoint_key="target_encoder"):
    print(f"Loading pretrained model from {pretrained}")
    checkpoint = robust_checkpoint_loader(pretrained, map_location="cpu")
    pretrained_dict = checkpoint[checkpoint_key]
    
    
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    # pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    for k, v in encoder.state_dict().items():
        if k not in pretrained_dict:
            print(f"key '{k}' could not be found in loaded state dict")
        elif pretrained_dict[k].shape != v.shape:
            print(f"{pretrained_dict[k].shape} | {v.shape}")
            print(f"key '{k}' is of different shape in model and loaded state dict")
            exit(1)
            pretrained_dict[k] = v
    msg = encoder.load_state_dict(pretrained_dict, strict=False)
    print(encoder)
    print(f"loaded pretrained model with msg: {msg}")
    print(f"loaded pretrained encoder from epoch: {checkpoint['epoch']}\n path: {pretrained}")
    del checkpoint
    return encoder

predictor = load_any_pretrained(predictor, pretrained, checkpoint_key="predictor")



Loading pretrained model from /data/wjl/vjepa2/ckpts/vitl.pt
PredictorMultiSeqWrapper(
  (backbone): VisionTransformerPredictor(
    (predictor_embed): Linear(in_features=1024, out_features=384, bias=True)
    (mask_tokens): ParameterList(
        (0): Parameter containing: [torch.float32 of size 1x1x384]
        (1): Parameter containing: [torch.float32 of size 1x1x384]
        (2): Parameter containing: [torch.float32 of size 1x1x384]
        (3): Parameter containing: [torch.float32 of size 1x1x384]
        (4): Parameter containing: [torch.float32 of size 1x1x384]
        (5): Parameter containing: [torch.float32 of size 1x1x384]
        (6): Parameter containing: [torch.float32 of size 1x1x384]
        (7): Parameter containing: [torch.float32 of size 1x1x384]
        (8): Parameter containing: [torch.float32 of size 1x1x384]
        (9): Parameter containing: [torch.float32 of size 1x1x384]
    )
    (predictor_blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): Layer

: 

In [48]:
predictor.state_dict().keys()

odict_keys(['backbone.predictor_embed.weight', 'backbone.predictor_embed.bias', 'backbone.mask_tokens.0', 'backbone.mask_tokens.1', 'backbone.predictor_blocks.0.norm1.weight', 'backbone.predictor_blocks.0.norm1.bias', 'backbone.predictor_blocks.0.attn.qkv.weight', 'backbone.predictor_blocks.0.attn.qkv.bias', 'backbone.predictor_blocks.0.attn.proj.weight', 'backbone.predictor_blocks.0.attn.proj.bias', 'backbone.predictor_blocks.0.norm2.weight', 'backbone.predictor_blocks.0.norm2.bias', 'backbone.predictor_blocks.0.mlp.fc1.weight', 'backbone.predictor_blocks.0.mlp.fc1.bias', 'backbone.predictor_blocks.0.mlp.fc2.weight', 'backbone.predictor_blocks.0.mlp.fc2.bias', 'backbone.predictor_blocks.1.norm1.weight', 'backbone.predictor_blocks.1.norm1.bias', 'backbone.predictor_blocks.1.attn.qkv.weight', 'backbone.predictor_blocks.1.attn.qkv.bias', 'backbone.predictor_blocks.1.attn.proj.weight', 'backbone.predictor_blocks.1.attn.proj.bias', 'backbone.predictor_blocks.1.norm2.weight', 'backbone.pred

In [49]:
pretrained_dict = robust_checkpoint_loader(pretrained, map_location="cpu")["predictor"]
pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
pretrained_dict.keys()

dict_keys(['backbone.predictor_embed.weight', 'backbone.predictor_embed.bias', 'backbone.mask_tokens.0', 'backbone.mask_tokens.1', 'backbone.mask_tokens.2', 'backbone.mask_tokens.3', 'backbone.mask_tokens.4', 'backbone.mask_tokens.5', 'backbone.mask_tokens.6', 'backbone.mask_tokens.7', 'backbone.mask_tokens.8', 'backbone.mask_tokens.9', 'backbone.predictor_blocks.0.norm1.weight', 'backbone.predictor_blocks.0.norm1.bias', 'backbone.predictor_blocks.0.attn.qkv.weight', 'backbone.predictor_blocks.0.attn.qkv.bias', 'backbone.predictor_blocks.0.attn.proj.weight', 'backbone.predictor_blocks.0.attn.proj.bias', 'backbone.predictor_blocks.0.norm2.weight', 'backbone.predictor_blocks.0.norm2.bias', 'backbone.predictor_blocks.0.mlp.fc1.weight', 'backbone.predictor_blocks.0.mlp.fc1.bias', 'backbone.predictor_blocks.0.mlp.fc2.weight', 'backbone.predictor_blocks.0.mlp.fc2.bias', 'backbone.predictor_blocks.1.norm1.weight', 'backbone.predictor_blocks.1.norm1.bias', 'backbone.predictor_blocks.1.attn.qkv

In [1]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader
from app.vjepa.utils import init_video_model
import copy

ckpts = robust_checkpoint_loader("/data/wjl/vjepa2/ckpts/vitl.pt", map_location="cpu")
ckpts.keys()




/home/wjl/.conda/envs/jepa_torch/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


dict_keys(['encoder', 'predictor', 'opt', 'scaler', 'target_encoder', 'epoch', 'loss', 'batch_size', 'world_size', 'lr'])